# Load dependencies

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from gitsource import GithubRepositoryDataReader
load_dotenv()

# TODO: add src parent folder to path
from src.data.index import build_index
from src.rag_helper import GitRAG

# Load documents

- First, we will pull the lesson pages straight from the course repository. We will use the commit `8c1834d` to make sure everyone works with the exact same data.
- `GithubRepositoryDataReader` downloads the entire repository and goes over all the files in it. Because we specify `allowed_extensions={"md"}`, it only checks the markdown files.
- We also pass a `filename_filter` so we don't grab every markdown file in the repo, like the top-level README. The lesson pages all live under a module's `lessons/` folder, so filtering on `/lessons/` keeps just those.
- Each file has a `parse()` method that returns a dictionary with its `filename` and `content`

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

# Q1. How many lesson pages

How many lesson pages are in the dataset?

-   24
-   72
-   240
-   720

In [ ]:
print(len(documents))

# Q2. Indexing and searching

Index the documents with minsearch - make `content` a text field and `filename` a keyword field. Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

-   `01-agentic-rag/lessons/03-rag.md`
-   `01-agentic-rag/lessons/14-agentic-loop.md`
-   `04-evaluation/lessons/13-llm-as-judge.md`
-   `06-best-practices/lessons/02-hybrid-search.md`

In [ ]:
print(documents[0].keys())

In [ ]:
index = build_index(
    documents=documents,
    text_fields=['content'],
    keyword_fields=['filename']
)

In [ ]:
query = "How does the agentic loop keep calling the model until it stops?"
search_result = index.search(query, num_results=1)
print(search_result[0]['filename'])

# Q3. RAG

Build a RAG over the index from Q2 and answer the query:

> How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

-   700
-   7000
-   70000
-   700000

In [ ]:
openai_client = OpenAI()
with open("templates/homework_01_instructions.txt", "r") as file:
    instructions = file.read().strip()
with open("templates/prompt_template.txt", "r") as file:
    prompt_template = file.read().strip()

In [ ]:
assistant = GitRAG(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
    prompt_template=prompt_template,
    search_boost_dict=None,
    search_filter_dict=None
)
question = "How does the agentic loop keep calling the model until it stops?"
answer = assistant.rag(question)

In [ ]:
# print(answer.output_text)
print(answer.usage.input_tokens)

# Q4. Chunking

The lesson pages are long - some are thousands of characters. Long documents make retrieval less precise: a match deep inside a page still pulls in the whole page. A common fix is chunking: split each page into smaller, overlapping pieces and index those instead.

gitsource has a helper for this: `chunk_documents`. It uses a sliding window - a window of `size` characters slides across the text in steps of `step` characters, and each window position becomes one chunk:

In [ ]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

With `size=2000` and `step=1000` (you can see the implementation [here](https://github.com/alexeygrigorev/gitsource/blob/master/gitsource/chunking.py)):

-   Each chunk is a window of `size` characters of the page.
-   The window moves forward by `step` characters between chunks. Since `step` is smaller than `size`, consecutive chunks overlap by `size - step` (1000) characters, so a passage split across a boundary still appears whole in one of the chunks.
-   Every chunk keeps the original fields (`filename`) and adds `start` (the offset in the page) and `content` (the chunk text).

How many chunks do you get?

-   70
-   295
-   1100
-   4500

In [ ]:
print(len(chunks))

In [ ]:
chunks[0].keys()

# Q5. RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the LLM. Let's measure that.

Index the chunks from Q4 (same as before: `content` as a text field, `filename` as a keyword field), point your RAG at the chunk index, and answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

-   about the same
-   3× fewer
-   10× fewer
-   30× fewer

In [ ]:
chunked_index = build_index(
    documents=chunks,
    text_fields=['content'],
    keyword_fields=['filename']
)

In [ ]:
assistant = GitRAG(
    index=chunked_index,
    llm_client=openai_client,
    instructions=instructions,
    prompt_template=prompt_template,
    search_boost_dict=None,
    search_filter_dict=None
)
question = "How does the agentic loop keep calling the model until it stops?"
answer = assistant.rag(question)

In [ ]:
print(answer.usage.input_tokens)

# Q6. Turning it into an agent

So far search runs once, with the exact query. Let's make it agentic: give the LLM a `search` tool and let it decide when (and what) to search.

- Create a `search` function that uses the chunk index. Give it a type hint and a docstring - most frameworks read them to build the tool schema for you.
- Build an agent with your `search` tool and run it.
- Use these instructions for the agent (they nudge it to search a few times):
> You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
- Ask it:
> How does the agentic loop work, and how is it different from plain RAG?

The agent decides on its own when to search and when to answer.

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback


def search(query: str) -> dict[str, str]:
    """
    Search the Course's Github Repo file chunks for entries matching the given query.
    """
    return chunked_index.search(query, num_results=5)

agent_tools = Tools()
agent_tools.add_tool(search)

In [ ]:
agnetic_rag_instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
"""
question = "How does the agentic loop work, and how is it different from plain RAG?"

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=agnetic_rag_instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [ ]:
result = runner.loop(
    prompt=question,
    callback=callback,
)

How many times did the agent call `search`?
> Note: the agent decides this itself, so it varies a little between runs - pick the closest option. We measured this with OpenAI `gpt-5.4-mini`; with a different model or provider the number may differ, so keep that in mind.

-   0
-   4
-   10
-   20